# OGIM India Data Extraction (Expanded)

This notebook extracts all available oil and gas infrastructure data specifically for India from the global `OGIM_v2.7.gpkg` database. It includes spatial filters for offshore fields and basins.

In [1]:
import geopandas as gpd
import pandas as pd
import os


In [2]:
gpkg_path = 'OGIM_v2.7.gpkg'

all_layers = [
    'Crude_Oil_Refineries',
    'Equipment_and_Components',
    'Gathering_and_Processing',
    'Injection_and_Disposal',
    'LNG_Facilities',
    'Natural_Gas_Compressor_Stations',
    'Natural_Gas_Flaring_Detections',
    'Offshore_Platforms',
    'Oil_Natural_Gas_Pipelines',
    'Oil_and_Natural_Gas_Basins',
    'Oil_and_Natural_Gas_Fields',
    'Oil_and_Natural_Gas_License_Blocks',
    'Oil_and_Natural_Gas_Wells',
    'Petroleum_Terminals',
    'Stations_Other',
    'Tank_Battery'
]

# Broad bounding box covering India and offshore EEZ (Arabian Sea, Bay of Bengal)
india_bbox = (66.0, 5.0, 98.0, 36.0)


In [3]:
os.makedirs('data', exist_ok=True)

for layer in all_layers:
    try:
        print(f"Processing {layer}...")
        # We use bounding box for spatial types that cross borders or exist offshore,
        # but for point infrastructures like wells/refineries, 'COUNTRY=INDIA' is highly accurate.
        if layer in ['Oil_and_Natural_Gas_Basins', 'Oil_and_Natural_Gas_Fields', 'Oil_Natural_Gas_Pipelines', 'Oil_and_Natural_Gas_License_Blocks', 'Offshore_Platforms']:
            gdf = gpd.read_file(gpkg_path, layer=layer, engine='pyogrio', bbox=india_bbox)
        else:
            gdf = gpd.read_file(gpkg_path, layer=layer, engine='pyogrio', where="COUNTRY='INDIA'")
            
        if not gdf.empty:
            output_path = f"data/{layer}_India.geojson"
            gdf.to_file(output_path, driver='GeoJSON')
            print(f"  => Saved {len(gdf)} features for {layer}")
        else:
            print(f"  => No data found for {layer}")
    except Exception as e:
        # If COUNTRY column doesn't exist, fallback to bbox.
        try:
            gdf = gpd.read_file(gpkg_path, layer=layer, engine='pyogrio', bbox=india_bbox)
            if not gdf.empty:
                output_path = f"data/{layer}_India.geojson"
                gdf.to_file(output_path, driver='GeoJSON')
                print(f"  => Saved {len(gdf)} features (spatial fallback) for {layer}")
        except Exception as e2:
            print(f"  => Error processing {layer}: {e2}")


Processing Crude_Oil_Refineries...
  => Saved 22 features for Crude_Oil_Refineries
Processing Equipment_and_Components...
  => No data found for Equipment_and_Components
Processing Gathering_and_Processing...
  => No data found for Gathering_and_Processing
Processing Injection_and_Disposal...
  => No data found for Injection_and_Disposal
Processing LNG_Facilities...
  => Saved 14 features for LNG_Facilities
Processing Natural_Gas_Compressor_Stations...
  => No data found for Natural_Gas_Compressor_Stations
Processing Natural_Gas_Flaring_Detections...
  => Saved 156 features for Natural_Gas_Flaring_Detections
Processing Offshore_Platforms...
  => Saved 6 features for Offshore_Platforms
Processing Oil_Natural_Gas_Pipelines...


  => Saved 187 features for Oil_Natural_Gas_Pipelines
Processing Oil_and_Natural_Gas_Basins...
  => Saved 49 features for Oil_and_Natural_Gas_Basins
Processing Oil_and_Natural_Gas_Fields...
  => Saved 184 features for Oil_and_Natural_Gas_Fields
Processing Oil_and_Natural_Gas_License_Blocks...
  => Saved 302 features for Oil_and_Natural_Gas_License_Blocks
Processing Oil_and_Natural_Gas_Wells...


  => No data found for Oil_and_Natural_Gas_Wells
Processing Petroleum_Terminals...
  => Saved 11 features for Petroleum_Terminals
Processing Stations_Other...
  => No data found for Stations_Other
Processing Tank_Battery...
  => No data found for Tank_Battery
